# Building a Context-Specific Salmonella GEM from RNA-seq Data Using iMATpy

**Author:** Gina Vazquez  
**Date:** March 2026  

## Goal
Integrate RNA-seq data into a Salmonella enterica subsp. enterica serovar Typhimurium str. LT2 genome-scale metabolic model (GEM) using the [iMATpy algorithm](https://imatpy.readthedocs.io/en/latest/).

## Dataset

## Notes

## Workflow
1. Load STM_V1_0 Model
3. Load thresholds tables
4. Extract the gene names from the threshold tables
5. Extract the gene names from the model
6. Compare gene IDs between RNA-seq and GEM
7. Subset the threshold data to have all model genes
8. Add values of zero for genes unassigned in threshold tables
9. Convert gene weights to reaction weights
10. Run iMatpy


# Load Packages

In [2]:
# core
import numpy as np
import pandas as pd

# COBRApy
from cobra.core.configuration import Configuration
from cobra.io import load_model
from cobra.core.solution import get_solution

#iMATpy
from imatpy.imat import imat, add_imat_constraints_
from imatpy.model_utils import read_model
from imatpy.parse_gpr import gene_to_rxn_weights
from imatpy.model_creation import generate_model

# Import the Base Model

[STM_V1_0 Model](http://bigg.ucsd.edu/models/STM_v1_0) is the most standard curated GEM for Salmonella and this specific strain. For more information on the curation of this model see [Bumann et al., 2011](https://link.springer.com/article/10.1186/1752-0509-5-8).

In [3]:
# load the test S.Tm LT2 GEM in cobrapy 
base_model = load_model("STM_v1_0")
base_model

Name,STM_v1_0
Memory address,7ecdc6a68910
Number of metabolites,1802
Number of reactions,2545
Number of genes,1271
Number of groups,0
Objective expression,1.0*BIOMASS_iRR1083_metals - 1.0*BIOMASS_iRR1083_metals_reverse_559d4
Compartments,"cytosol, periplasm, extracellular space"


# Load RNA-seq Gene Expression

In [4]:
# read in the csv files of the threshold rna-seq data
stm_threshold_10_90 = pd.read_csv("/home/gmvaz/2026_GEMs/imat_prep_test/stm_threshold_10_90.csv")
stm_threshold_15_85 = pd.read_csv("/home/gmvaz/2026_GEMs/imat_prep_test/stm_threshold_15_85.csv")
stm_threshold_25_75 = pd.read_csv("/home/gmvaz/2026_GEMs/imat_prep_test/stm_threshold_25_75.csv")
mixed_stm_threshold_10_90 = pd.read_csv("/home/gmvaz/2026_GEMs/imat_prep_test/mixed_stm_threshold_10_90.csv")
mixed_stm_threshold_15_85 = pd.read_csv("/home/gmvaz/2026_GEMs/imat_prep_test/mixed_stm_threshold_15_85.csv")
mixed_stm_threshold_25_75 = pd.read_csv("/home/gmvaz/2026_GEMs/imat_prep_test/mixed_stm_threshold_25_75.csv")

#print an example of one of the files
print(stm_threshold_10_90)

        genes  threshold_category
0     STM0001                   0
1     STM0002                   0
2     STM0003                   0
3     STM0004                   0
4     STM0005                   0
...       ...                 ...
4666  PSLT106                  -1
4667  PSLT107                  -1
4668  PSLT108                  -1
4669  PSLT110                  -1
4670  PSLT111                  -1

[4671 rows x 2 columns]


# Extract the gene names from one of the CSV Files

Here I am extracting the gene names used in the threshold CSV files to cross-reference them the genes in the model. 
I am checking that: 
1) The genes have the same naming convention in both the model and threshold CSVs.
2) Most genes in the CSV files overlap with those in the model.

In [10]:
# python set
threshold_genes = set(stm_threshold_10_90["genes"].astype(str).str.strip())

# look at the first 10 in the set
list(threshold_genes)[:10]

['STM2187',
 'STM0576',
 'STM4332',
 'STM0412',
 'STM1569',
 'STM1142',
 'STM2183',
 'STM3072',
 'STM1672',
 'STM4212']

# Extract the gene names from the model

In [11]:
# First, look at the components of a model to figure out "where" the gene_id's are
dir(base_model)

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__enter__',
 '__eq__',
 '__exit__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_annotation',
 '_compartments',
 '_contexts',
 '_id',
 '_populate_solver',
 '_repr_html_',
 '_sbml',
 '_set_id_with_model',
 '_solver',
 '_tolerance',
 'add_boundary',
 'add_cons_vars',
 'add_groups',
 'add_metabolites',
 'add_reactions',
 'annotation',
 'boundary',
 'compartments',
 'constraints',
 'copy',
 'demands',
 'exchanges',
 'genes',
 'get_associated_groups',
 'groups',
 'id',
 'medium',
 'merge',
 'metabolites',
 'name',
 'notes',
 'objective',
 'objective_direction',
 'optimize',
 'problem',
 'reactions',
 'remove_cons_vars',
 'remove_groups',
 'remo

The 'genes' attribute looks like it will have the gene_id's. 

In [19]:
# print the amount of genes in the model
print(len(base_model.genes))
# inspect the first five genes to see how they are named
base_model.genes[0:4]

1271


[<Gene STM0685 at 0x7ecdc65ca110>,
 <Gene STM1203 at 0x7ecdc657c290>,
 <Gene STM4122 at 0x7ecdc65cb050>,
 <Gene STM1473 at 0x7ecdc65cb290>]

The gene identifier for the first five genes in the model.genes attribute have the same naming convention as the CSV files. Let's further inspect gene attributes.

In [20]:
# the attributes available for the first gene in the list
dir(base_model.genes[0])

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_annotation',
 '_functional',
 '_id',
 '_model',
 '_reaction',
 '_repr_html_',
 '_set_id_with_model',
 'annotation',
 'copy',
 'functional',
 'id',
 'knock_out',
 'model',
 'name',
 'notes',
 'reactions']

In [23]:
# we can print the gene id, name, and associated reactions for the first gene in the list
print(base_model.genes[0].id)
print(base_model.genes[0].name)
print(base_model.genes[0].reactions)

STM0685
nagE
frozenset({<Reaction ACGAptspp at 0x7ecdc63c9a10>})


The gene_ID's are in the model.genes object. Let's extract these into a set of gene IDs from the model. 

In [35]:
# make a list of genes and save it as a set 
model_genes = set([g.id for g in base_model.genes])

# look at a few of the genes in the list 
print(list(model_genes)[:5])

# check the length of the set
print(len(model_genes))

['STM2183', 'STM1569', 'STM2306', 'STM1469', 'STM0039']
1271


# Compare Gene IDs Between RNA-seq and GEM

In [37]:
# find genes that exist in the model and RNA-seq
overlap = threshold_genes.intersection(model_genes)

# find genes that appear in the RNA-Seq data but are not in the model
only_stm = threshold_genes.difference(model_genes)

# find genes that exist in the model but do not appear in RNA-Seq data
only_model = model_genes.difference(threshold_genes)

Genes that overlap between the model and RNA-Seq data will be used to constrain the model. I expect to see very high overlap between the two. Otherwise, it would suggest an issue with the gene names argument used when counting genes or (less likely) the wrong strain GEM is being used. 

Some genes may appear only in the RNA-Sequence data but not the GEM. This could be caused by:
1) the gene not being included in the model since the model is only of primary metabolism
2) annotation mismatch
3) different genome version
4) typos
These genes won't affect the GEM  since they aren't present. If they are important, we'll need to manually add them to the model. 

Som genes may exist in the GEM but may not appear in the RNA-Seq data. This could be caused by:
1) genes are not expressed
2) genes were filtered out in the RNA-Sequence preprocessing
3) genes were not detected in the RNA-Seq.
These will be treated as unchanged/unaltered in the model though it's worth evaluating which genes weren't expressed and if that's to be expected. 

In [39]:
print("Genes in RNA-seq:", len(threshold_genes))
print("Genes in model:", len(model_genes))
print("Overlap:", len(overlap))
print("Only in RNA-seq:", len(only_stm))
print("Only in model:", len(only_model))

Genes in RNA-seq: 4671
Genes in model: 1271
Overlap: 1251
Only in RNA-seq: 3420
Only in model: 20


Let's look at the 20 genes that are in the model but not present.

In [52]:
# get the gene id and gene name of every gene object in model.genes if the gene id is present in only_model
missing_genes = [
    (g.id, g.name)
    for g in base_model.genes
    if g.id in only_model
]
# not sure what to make of this, these genes don't have standard names 
print(missing_genes)

# look up a gene by ID to see what is the name and associated reactions
gene1 = base_model.genes.get_by_id("STM4188_S")

print(gene1.id)
print(gene1.name)
print(gene1.reactions)

[('s0001', 's0001'), ('STM1464', 'STM1464'), ('STM2975', 'STM2975'), ('STM1465', 'STM1465'), ('STM4454', 'STM4454'), ('STM1746_S', 'STM1746_S'), ('STM2838_S', 'STM2838_S'), ('STM3290_S', 'STM3290_S'), ('STM4383_S', 'STM4383_S'), ('STM4188_S', 'STM4188_S'), ('STM2323_S', 'STM2323_S'), ('STM2316_S', 'STM2316_S'), ('STM4580_S', 'STM4580_S'), ('STM4559_S', 'STM4559_S'), ('STM4278_S', 'STM4278_S'), ('STM2499_S', 'STM2499_S'), ('STM1892_S', 'STM1892_S'), ('STM2105_S', 'STM2105_S'), ('STM4305_S', 'STM4305_S'), ('STM4424_S', 'STM4424_S')]
STM4188_S
STM4188_S
frozenset({<Reaction METS at 0x7ecdc5616010>})


In [54]:
# Let's look in the GEM not found in the RNA-Seq
list(only_model)

['STM1746_S',
 'STM4383_S',
 'STM2838_S',
 'STM4424_S',
 'STM1892_S',
 'STM4559_S',
 'STM2499_S',
 'STM2316_S',
 'STM3290_S',
 'STM4188_S',
 'STM1465',
 's0001',
 'STM4454',
 'STM2323_S',
 'STM2975',
 'STM1464',
 'STM4305_S',
 'STM4580_S',
 'STM4278_S',
 'STM2105_S']

# Filter the RNA-Seq threshold data to only keep the genes present in the model. 
The 20 genes that are present in the model but not in the RNA-Seq data will be unaltered for now. 

In [57]:
# subset the rna-seq expression data to only have the genes present  in the model

stm_threshold_10_90_model = stm_threshold_10_90[stm_threshold_10_90["genes"].isin(model_genes)]
stm_threshold_15_85_model = stm_threshold_15_85[stm_threshold_15_85["genes"].isin(model_genes)]
stm_threshold_25_75_model = stm_threshold_25_75[stm_threshold_25_75["genes"].isin(model_genes)]

mixed_stm_threshold_10_90_model = mixed_stm_threshold_10_90[mixed_stm_threshold_10_90["genes"].isin(model_genes)]
mixed_stm_threshold_15_85_model = mixed_stm_threshold_15_85[mixed_stm_threshold_15_85["genes"].isin(model_genes)]
mixed_stm_threshold_25_75_model = mixed_stm_threshold_25_75[mixed_stm_threshold_25_75["genes"].isin(model_genes)]

In [59]:
# to these subsets add the 20 genes present in the model but missing in the rna-seq data and assign a value of 0
# convert from dataframe to panda series, make sure all the genes from the model exist, and fill missing values with 0

stm_10_90_expr = stm_threshold_10_90_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)
stm_15_85_expr = stm_threshold_15_85_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)
stm_25_75_expr = stm_threshold_25_75_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)

mixed_10_90_expr = mixed_stm_threshold_10_90_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)
mixed_15_85_expr = mixed_stm_threshold_15_85_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)
mixed_25_75_expr = mixed_stm_threshold_25_75_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)

len(stm_10_90_expr)

1271

# Convert Expression Values to iMATpy Reaction Weights

In [60]:
stm_10_90_rxn = gene_to_rxn_weights(base_model, stm_10_90_expr)
stm_15_85_rxn = gene_to_rxn_weights(base_model, stm_15_85_expr)
stm_25_75_rxn = gene_to_rxn_weights(base_model, stm_25_75_expr)

mixed_10_90_rxn = gene_to_rxn_weights(base_model, mixed_10_90_expr)
mixed_15_85_rxn = gene_to_rxn_weights(base_model, mixed_15_85_expr)
mixed_25_75_rxn = gene_to_rxn_weights(base_model, mixed_25_75_expr)

# Run iMAT

## Select the Solver

Set solver to glpk since this is the default optimizer installed with Cobrapy. Other solver options are cplex and gurobi. [Daniel Machado, 2024](https://journals.asm.org/doi/10.1128/msystems.00833-23) has benchmarking paper on optimization solvers for GEMs.

In [ ]:
%%file x.py
Configuration().solver = "glpk"

In [62]:
# make a dictionary of reaction weights
reaction_weights = {
    "stm_10_90": stm_10_90_rxn,
    "stm_15_85": stm_15_85_rxn,
    "stm_25_75": stm_25_75_rxn,
    "mixed_10_90": mixed_10_90_rxn,
    "mixed_15_85": mixed_15_85_rxn,
    "mixed_25_75": mixed_25_75_rxn
}

# code to run one at a time
# model_stm_10_90 = imat(base_model, stm_10_90_rxn)

In [ ]:
context_models = {}

for name, rxn_weights in reaction_weights.items():
    
    print(f"Running iMAT for {name}")
    
    context_model = imat(base_model, rxn_weights)
    
    context_models[name] = context_model

# Run Flux Balance Analysis